# Đồng bộ dữ liệu và kết quả với Hugging Face

Notebook này dùng cho repo `mdn_trajectory_forecasting`. Nó chỉ đồng bộ các thư mục đang bị Git bỏ qua (`data/`, `results/`, `slides/`) lên dataset Hugging Face, không đẩy `.git/`, `.venv/` hoặc mã nguồn.

Sau khi tải, cấu trúc thư mục được khôi phục như sau:
```text
mdn_trajectory_forecasting/
├── data/
├── results/
└── slides/
```

In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'])

from pathlib import Path
import os

from huggingface_hub import login, upload_folder, snapshot_download

## 1. Xác định đúng thư mục repo

Không hard-code `/home/...`. Cell này tìm thư mục repo từ thư mục notebook hiện tại hoặc các thư mục cha, nên thành viên có thể chạy trên máy khác mà không sửa đường dẫn.

In [ ]:
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'base_mdn').is_dir():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy repo. Hãy mở notebook từ bên trong mdn_trajectory_forecasting.'
    )

REPO_ROOT = find_repo_root()
HF_REPO_ID = 'quinha10/HTTM-GK-2026'
HF_REPO_TYPE = 'dataset'
SYNC_DIRS = ('data', 'results', 'slides')

print('Repo:', REPO_ROOT)
print('Hugging Face:', f'https://huggingface.co/datasets/{HF_REPO_ID}')
for name in SYNC_DIRS:
    path = REPO_ROOT / name
    print(f'{name}: {path} | tồn tại={path.exists()}')

## 2. Kiểm tra những gì sẽ upload

Các file `.DS_Store`, cache Python và file rỗng tạm thời sẽ bị bỏ qua. Tổng dung lượng có thể lớn (dữ liệu/kết quả hiện tại khoảng 1.45 GB), vì vậy cần chạy cell upload một lần trên máy có mạng ổn định.

In [ ]:
def inventory():
    rows = []
    for name in SYNC_DIRS:
        root = REPO_ROOT / name
        files = [p for p in root.rglob('*') if p.is_file()] if root.exists() else []
        size = sum(p.stat().st_size for p in files)
        rows.append((name, len(files), size / (1024 ** 3)))
    return rows

for name, count, size_gb in inventory():
    print(f'{name:8s} | {count:6d} files | {size_gb:8.3f} GiB')

## 3. Upload từ máy đã có dữ liệu

Chỉ chạy cell này trên máy đang chứa dữ liệu/kết quả đầy đủ. `login()` sẽ mở luồng đăng nhập; không ghi token vào notebook.

In [ ]:
login()

IGNORE_PATTERNS = [
    '**/.DS_Store',
    '**/__pycache__/**',
    '**/*.pyc',
    '**/train_loss 2.png',
]

for name in SYNC_DIRS:
    local_dir = REPO_ROOT / name
    if not local_dir.exists():
        print(f'Bỏ qua {name}: không tồn tại')
        continue
    print(f'Đang upload {name}/ ...')
    upload_folder(
        folder_path=str(local_dir),
        path_in_repo=name,
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        ignore_patterns=IGNORE_PATTERNS,
        commit_message=f'Sync {name} from repository',
    )
    print(f'Đã upload {name}/')

print('Hoàn tất upload:', f'https://huggingface.co/datasets/{HF_REPO_ID}/tree/main')

## 4. Tải về đúng vị trí repo cho mọi thành viên

Thành viên clone GitHub repo trước, mở notebook này từ bên trong repo, đăng nhập Hugging Face rồi chạy cell dưới. `snapshot_download(..., local_dir=REPO_ROOT)` giữ nguyên `data/`, `results/`, `slides/`; không cần sửa đường dẫn. Nếu file đã tồn tại, Hugging Face chỉ tải phần thiếu/thay đổi.

In [ ]:
login()

downloaded = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    local_dir=str(REPO_ROOT),
    allow_patterns=[
        'data/**',
        'results/**',
        'slides/**',
    ],
    ignore_patterns=['**/.DS_Store', '**/__pycache__/**', '**/*.pyc'],
)
print('Đã tải vào:', downloaded)
for name in SYNC_DIRS:
    path = REPO_ROOT / name
    print(f'{name}: tồn tại={path.exists()}')

### Lưu ý

- Dataset Hugging Face phải là public hoặc tài khoản của thành viên phải có quyền truy cập.
- Không commit token vào GitHub/notebook.
- Nếu chỉ muốn tải dữ liệu mà không tải slide, có thể sửa `allow_patterns` thành `['data/**', 'results/**']`; mặc định notebook giữ cả ba thư mục để khớp repo hiện tại.